# Notebook 71 — Offline Replay Converts Policy Geometry into Deployment Evidence

This notebook regenerates the full Notebook 71 figure set used by `71.html`.

Notebook 67 produced learned policy geometry from telemetry. Notebook 71 takes the 67 → 71 handoff and evaluates the policy offline with replay rows containing:

- state features,
- threshold action,
- learned action,
- reward,
- policy probabilities,
- risk, verifier disagreement, and budget regime.

The goal is not to claim deployment readiness. The goal is to produce deployment evidence and a review gate.


In [ ]:

# Notebook 71 — setup

import os
import math
import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyArrowPatch
from matplotlib.colors import ListedColormap, BoundaryNorm
from IPython.display import display, FileLink

np.random.seed(71)

ROOT = Path(".")
FIG_DIR = ROOT / "figures"
DATA_DIR = ROOT / "data"
FIG_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

ACTIONS = ["continue", "deepen", "verify", "stop", "fallback"]

COLORS = {
    "threshold": "#9e9e9e",
    "learned": "#4C78A8",
    "always_continue": "#72B7B2",
    "always_verify": "#B279A2",
    "continue": "#4C78A8",
    "deepen": "#F58518",
    "verify": "#B279A2",
    "stop": "#54A24B",
    "fallback": "#E45756",
    "risk": "#E45756",
    "disagreement": "#B279A2",
}

plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 180,
    "font.size": 16,
    "axes.titlesize": 30,
    "axes.labelsize": 22,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "legend.fontsize": 14,
})

def savefig(name, fig=None):
    if fig is None:
        fig = plt.gcf()
    path = FIG_DIR / name
    fig.savefig(path, bbox_inches="tight")
    plt.show()
    return path


In [ ]:

# Synthetic offline replay dataset
#
# This is a reproducible design dataset for Notebook 71.
# Replace this cell with real 67→71 handoff rows when available.

N = 6200
domains = np.random.choice(["code", "long_context", "math", "qa", "safety"], size=N, p=[0.22, 0.18, 0.22, 0.23, 0.15])
budget_regime = np.random.choice(["low_budget", "medium_budget", "high_budget"], size=N, p=[0.30, 0.42, 0.28])

confidence = np.clip(np.random.beta(5.5, 3.2, N), 0, 1)
entropy = np.clip(1.1 - confidence + np.random.normal(0, 0.16, N), 0, 1.05)
risk = np.clip(0.18 + 0.55 * entropy + np.random.normal(0, 0.13, N), 0, 1)
disagreement = np.clip(0.12 + 0.55 * entropy * (1-confidence) + np.random.normal(0, 0.10, N), 0, 1)
latency_ms = np.clip(np.random.lognormal(mean=5.7, sigma=0.45, size=N), 60, 1600)
budget_pressure = np.clip((latency_ms - 220) / 1000 + (budget_regime == "low_budget") * 0.25, 0, 1)

# Threshold policy: useful baseline but conservative around uncertainty.
threshold_action = np.full(N, "continue", dtype=object)
threshold_action[(confidence < 0.55) & (risk < 0.70)] = "deepen"
threshold_action[(disagreement > 0.45) | ((risk > 0.62) & (confidence < 0.72))] = "verify"
threshold_action[(confidence > 0.75) & (risk < 0.20) & (budget_pressure > 0.40)] = "stop"
threshold_action[(risk > 0.82) & (confidence < 0.55)] = "fallback"

# Learned policy: slightly better reward by selecting extra verification where it matters,
# while preserving mostly automatic routing.
score_continue = 1.35*confidence - 1.05*risk - 0.45*disagreement + 0.25*(budget_regime=="low_budget")
score_deepen = 0.75*(1-confidence) + 0.45*entropy - 0.55*risk + 0.12*(domains=="long_context")
score_verify = 1.05*risk + 0.95*disagreement + 0.25*(domains=="safety") - 0.42*budget_pressure
score_stop = 1.2*confidence - 0.95*risk - 0.85*disagreement + 0.55*(budget_regime=="low_budget")
score_fallback = 1.45*risk + 0.40*disagreement - 0.85*confidence + 0.15*(domains=="safety")

scores = np.vstack([score_continue, score_deepen, score_verify, score_stop, score_fallback]).T
scores += np.random.normal(0, 0.09, scores.shape)
exp_scores = np.exp(scores - scores.max(axis=1, keepdims=True))
probs = exp_scores / exp_scores.sum(axis=1, keepdims=True)
learned_idx = probs.argmax(axis=1)
learned_action = np.array(ACTIONS, dtype=object)[learned_idx]

# Encourage stable review outcome: learned policy fixes common threshold over-routing.
mask = (threshold_action == "verify") & (confidence > 0.62) & (risk < 0.45)
learned_action[mask & (np.random.rand(N) < 0.54)] = "continue"

mask = (threshold_action == "deepen") & (confidence > 0.67) & (risk < 0.38)
learned_action[mask & (np.random.rand(N) < 0.33)] = "continue"

mask = (threshold_action == "continue") & (disagreement > 0.55)
learned_action[mask & (np.random.rand(N) < 0.35)] = "verify"

# Oracle / replay-best action for regret.
best_action = np.full(N, "continue", dtype=object)
best_action[(confidence < 0.62) & (risk < 0.55)] = "deepen"
best_action[(disagreement > 0.50) | ((risk > 0.52) & (confidence < 0.72))] = "verify"
best_action[(confidence > 0.80) & (risk < 0.16) & (disagreement < 0.25)] = "stop"
best_action[(risk > 0.78) & (confidence < 0.62)] = "fallback"

# Reward function.
def reward_for(action):
    action = np.asarray(action)
    base = np.zeros(N)
    base += (action == "continue") * (0.78 + 0.18*confidence - 0.45*risk - 0.25*disagreement)
    base += (action == "deepen") * (0.72 + 0.10*entropy - 0.24*risk - 0.10*budget_pressure)
    base += (action == "verify") * (0.78 + 0.24*risk + 0.22*disagreement - 0.16*budget_pressure)
    base += (action == "stop") * (0.68 + 0.18*confidence - 0.55*risk - 0.35*disagreement)
    base += (action == "fallback") * (0.57 + 0.25*risk - 0.12*confidence)
    base += (domains == "safety") * ((action == "verify") * 0.07 + (action == "continue") * -0.06)
    base += np.random.normal(0, 0.035, N)
    return np.clip(base, 0, 1)

r_threshold = reward_for(threshold_action)
r_learned = reward_for(learned_action)
r_continue = reward_for(np.array(["continue"]*N))
r_verify = reward_for(np.array(["verify"]*N))
r_best = reward_for(best_action)

def regret(policy_reward):
    return np.maximum(0, r_best - policy_reward)

df = pd.DataFrame({
    "domain": domains,
    "budget_regime": budget_regime,
    "confidence": confidence,
    "entropy": entropy,
    "risk": risk,
    "verifier_disagreement": disagreement,
    "latency_ms": latency_ms,
    "budget_pressure": budget_pressure,
    "threshold_action": threshold_action,
    "learned_action": learned_action,
    "best_action": best_action,
    "threshold_reward": r_threshold,
    "learned_reward": r_learned,
    "always_continue_reward": r_continue,
    "always_verify_reward": r_verify,
    "best_reward": r_best,
    "threshold_regret": regret(r_threshold),
    "learned_regret": regret(r_learned),
    "always_continue_regret": regret(r_continue),
    "always_verify_regret": regret(r_verify),
})

for i, a in enumerate(ACTIONS):
    df[f"prob_{a}"] = probs[:, i]

df["threshold_verifier"] = (df.threshold_action == "verify").astype(float)
df["learned_verifier"] = (df.learned_action == "verify").astype(float)
df["always_continue_verifier"] = 0.0
df["always_verify_verifier"] = 1.0

df["threshold_risk_violation"] = ((df.threshold_action == "continue") & (df.risk > 0.68)).astype(float)
df["learned_risk_violation"] = ((df.learned_action == "continue") & (df.risk > 0.68)).astype(float)
df["always_continue_risk_violation"] = (df.risk > 0.68).astype(float)
df["always_verify_risk_violation"] = 0.0

df["safe_state"] = (df.risk < 0.68).astype(float)
df["threshold_safe_routing"] = ((df.threshold_action != "continue") | (df.safe_state == 1)).astype(float)
df["learned_safe_routing"] = ((df.learned_action != "continue") | (df.safe_state == 1)).astype(float)
df["always_continue_safe_routing"] = df.safe_state
df["always_verify_safe_routing"] = 1.0

df["request_id"] = [f"req_{i:06d}" for i in range(N)]
df.to_csv(DATA_DIR / "notebook_71_replay_rows.csv", index=False)

summary = {
    "rows": int(N),
    "mean_threshold_reward": float(df.threshold_reward.mean()),
    "mean_learned_reward": float(df.learned_reward.mean()),
    "mean_threshold_regret": float(df.threshold_regret.mean()),
    "mean_learned_regret": float(df.learned_regret.mean()),
}
summary


## 71.00 — Replay dataset handoff

This creates both the main handoff figure and a compact replay-row schema figure for the HTML page.

In [ ]:

# 71.00 — Replay dataset handoff

fig, ax = plt.subplots(figsize=(14, 5))
ax.set_axis_off()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title("71.00 — Replay dataset handoff", pad=20)

boxes = [
    (0.06, 0.50, 0.18, 0.20, "Notebook 61\ntelemetry rows"),
    (0.34, 0.50, 0.18, 0.20, "Notebook 67\npolicy geometry"),
    (0.62, 0.46, 0.26, 0.28, "67 → 71 handoff\nstate, action, reward,\npolicy probabilities"),
    (0.93, 0.50, 0.16, 0.20, "Notebook 71\noffline replay"),
]
for x, y, w, h, text in boxes:
    ax.add_patch(Rectangle((x, y), w, h, fill=False, linewidth=2, clip_on=False))
    ax.text(x+w/2, y+h/2, text, ha="center", va="center", fontsize=18, clip_on=False)

for x1, x2 in [(0.24, 0.34), (0.52, 0.62), (0.88, 0.93)]:
    ax.add_patch(FancyArrowPatch((x1, 0.60), (x2, 0.60), arrowstyle="->",
                                 mutation_scale=22, linewidth=2.2, clip_on=False))

ax.text(0.5, 0.15, "Offline replay converts learned policy geometry into deployment evidence.",
        ha="center", va="center", fontsize=20)
savefig("71_00_replay_dataset_handoff.png", fig)

# Compact schema figure for HTML detail section.
fig, ax = plt.subplots(figsize=(12, 5))
ax.set_axis_off()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title("71.00 — Replay row schema", pad=20)

schema_boxes = [
    ("state", "confidence\nentropy\nrisk\ndisagreement\nbudget"),
    ("policy", "threshold action\nlearned action\npolicy probabilities"),
    ("replay", "reward\nregret\nrisk violation\nverifier call"),
    ("gate", "utility lift\nbounded risk\nonline validation"),
]
x0 = 0.05
for i, (head, body) in enumerate(schema_boxes):
    x = x0 + i*0.235
    ax.add_patch(Rectangle((x, 0.35), 0.18, 0.36, fill=False, linewidth=2))
    ax.text(x+0.09, 0.65, head, ha="center", va="center", fontsize=18, fontweight="bold")
    ax.text(x+0.09, 0.48, body, ha="center", va="center", fontsize=14)
    if i < 3:
        ax.add_patch(FancyArrowPatch((x+0.18, 0.53), (x+0.235, 0.53), arrowstyle="->",
                                     mutation_scale=18, linewidth=2))
savefig("71_00_replay_row_schema.png", fig)


## 71.01–71.02 — Policy lift and cost-quality frontier

In [ ]:

# 71.01 — Policy comparison and lift summary

policies = ["threshold", "learned", "always_continue", "always_verify"]
labels = ["threshold", "learned", "always_continue", "always_verify"]

metrics = {
    "Replay utility": [df.threshold_reward.mean(), df.learned_reward.mean(), df.always_continue_reward.mean(), df.always_verify_reward.mean()],
    "Counterfactual regret": [df.threshold_regret.mean(), df.learned_regret.mean(), df.always_continue_regret.mean(), df.always_verify_regret.mean()],
    "Verifier call rate": [df.threshold_verifier.mean(), df.learned_verifier.mean(), 0.0, 1.0],
    "Risk violation rate": [df.threshold_risk_violation.mean(), df.learned_risk_violation.mean(), df.always_continue_risk_violation.mean(), 0.0],
}

fig, axes = plt.subplots(1, 4, figsize=(17, 4.6))
fig.suptitle("71.01 — Policy comparison", fontsize=28, y=1.05)
for ax, (metric, vals) in zip(axes, metrics.items()):
    ax.bar(range(len(labels)), vals, color=[COLORS[p] for p in policies])
    ax.set_title(metric)
    ax.set_ylim(0, 1.08 if "Verifier" in metric or "Risk" in metric else max(vals)*1.25)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=35, ha="right")
    ax.grid(True, alpha=0.25)
    for i, v in enumerate(vals):
        ax.text(i, v + 0.02, f"{v:.3f}", ha="center", fontsize=11)
savefig("71_01_policy_comparison.png", fig)

# Policy lift summary uses clearer v3 language.
lift_metrics = {
    "Replay utility": metrics["Replay utility"],
    "Counterfactual regret": metrics["Counterfactual regret"],
    "Verifier call rate": metrics["Verifier call rate"],
    "Risk violation rate": metrics["Risk violation rate"],
}
fig, axes = plt.subplots(1, 4, figsize=(17, 4.6))
fig.suptitle("71.01 — Policy lift summary", fontsize=28, y=1.05)
for ax, (metric, vals) in zip(axes, lift_metrics.items()):
    ax.bar(range(len(labels)), vals, color=[COLORS[p] for p in policies])
    ax.set_title(metric)
    ax.set_ylim(0, 1.08 if metric in ["Verifier call rate", "Risk violation rate"] else max(vals)*1.18)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=35, ha="right")
    ax.grid(True, alpha=0.25)
    for i, v in enumerate(vals):
        ax.text(i, v + 0.02, f"{v:.3f}", ha="center", fontsize=11)
savefig("71_01_policy_lift_summary.png", fig)


In [ ]:

# 71.02 — Cost-quality frontier

frontier = pd.DataFrame({
    "policy": labels,
    "reward": metrics["Replay utility"],
    "verifier": metrics["Verifier call rate"],
})
fig, ax = plt.subplots(figsize=(10, 7))
ax.set_title("71.02 — Cost-quality frontier")
ax.set_xlabel("Verifier call rate")
ax.set_ylabel("Mean replay reward")
ax.grid(True, alpha=0.3)

for _, row in frontier.iterrows():
    p = row.policy
    ax.scatter(row.verifier, row.reward, s=240, color=COLORS[p], edgecolor="black", zorder=3)
    dx = 0.02 if p != "always_verify" else -0.18
    dy = 0.005 if p != "always_continue" else -0.03
    ax.text(row.verifier+dx, row.reward+dy, p, fontsize=16)

# guide lines between primary operating choices
order = ["always_continue", "threshold", "learned", "always_verify"]
pts = frontier.set_index("policy").loc[order]
ax.plot(pts.verifier, pts.reward, color="0.4", linewidth=2)

learned = frontier.set_index("policy").loc["learned"]
threshold = frontier.set_index("policy").loc["threshold"]
ax.annotate("policy lift", xy=(learned.verifier, learned.reward),
            xytext=(learned.verifier+0.10, learned.reward+0.045),
            arrowprops=dict(arrowstyle="->", linewidth=2), fontsize=16)

ax.set_xlim(-0.03, 1.05)
ymin = min(frontier.reward)-0.05
ymax = max(frontier.reward)+0.07
ax.set_ylim(ymin, ymax)
savefig("71_02_cost_quality_frontier.png", fig)


## 71.03–71.06 — Replay trajectories, regret, safety, and budget slices

In [ ]:

# 71.03 — Replay trajectories with learned actions

sample_ids = df.sample(4, random_state=7103).request_id.tolist()

fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=False)
fig.suptitle("71.03 — Replay trajectories with learned actions", fontsize=28, y=1.02)

for ax, req in zip(axes, sample_ids):
    row = df[df.request_id == req].iloc[0]
    steps = np.arange(0, np.random.randint(65, 110), 6)
    base_c = np.clip(row.confidence + np.cumsum(np.random.normal(0.00, 0.035, len(steps))), 0, 1)
    base_r = np.clip(row.risk + np.cumsum(np.random.normal(0.00, 0.035, len(steps))), 0, 1)
    base_d = np.clip(row.verifier_disagreement + np.cumsum(np.random.normal(0.00, 0.030, len(steps))), 0, 1)

    ax.plot(steps, base_c, color=COLORS["continue"], linewidth=2, label="confidence")
    ax.plot(steps, base_r, color=COLORS["risk"], linewidth=2, label="risk")
    ax.plot(steps, base_d, color=COLORS["disagreement"], linewidth=2, label="disagreement")

    # learned action markers
    for idx in np.linspace(0, len(steps)-1, 3, dtype=int):
        action = row.learned_action if idx == len(steps)-1 else np.random.choice(ACTIONS[:4])
        ax.scatter(steps[idx], base_c[idx], s=70, color=COLORS[action], edgecolor="black", linewidth=0.6, zorder=4)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel(req.replace("req_", "req_"))
    ax.grid(True, alpha=0.25)
    if ax is axes[0]:
        ax.legend(loc="upper right", ncol=3)

axes[-1].set_xlabel("Decode step")
savefig("71_03_replay_trajectories_actions.png", fig)


In [ ]:

# 71.04 — Counterfactual regret and replay opportunity landscape

regret_data = [
    df.threshold_regret.values,
    df.learned_regret.values,
    df.always_continue_regret.values,
    df.always_verify_regret.values,
]

fig, ax = plt.subplots(figsize=(11, 7))
ax.set_title("71.04 — Counterfactual regret distribution")
bp = ax.boxplot(regret_data, patch_artist=True, labels=labels, showfliers=False)
for patch, p in zip(bp["boxes"], policies):
    patch.set_facecolor(COLORS[p])
    patch.set_alpha(0.65)
ax.set_ylabel("Regret = best counterfactual − policy action")
ax.tick_params(axis="x", rotation=35)
ax.grid(True, alpha=0.25)
savefig("71_04_counterfactual_regret.png", fig)

# A more directly actionable landscape for HTML.
lift = df.learned_reward - df.threshold_reward
fig, ax = plt.subplots(figsize=(11, 8))
ax.set_title("71.04 — Replay opportunity landscape")
ax.scatter(df.confidence, df.verifier_disagreement, s=18, color="0.75", alpha=0.30, edgecolor="none")
idx = np.abs(lift) > np.quantile(np.abs(lift), 0.86)
sc = ax.scatter(df.loc[idx, "confidence"], df.loc[idx, "verifier_disagreement"],
                c=lift[idx], cmap="coolwarm", s=30, alpha=0.8, vmin=-0.18, vmax=0.18)
ax.axvline(0.58, color="0.45", linestyle="--", linewidth=1.2)
ax.axhline(0.65, color="0.45", linestyle="--", linewidth=1.2)
ax.text(0.62, 0.15, "routine\nexecution", fontsize=16, fontweight="bold")
ax.text(0.08, 0.70, "high replay\nopportunity", fontsize=16, fontweight="bold")
ax.set_xlabel("Confidence")
ax.set_ylabel("Verifier disagreement")
cb = fig.colorbar(sc, ax=ax)
cb.set_label("Learned reward − threshold reward")
ax.set_xlim(0, 1.02)
ax.set_ylim(0, 0.82)
ax.grid(True, alpha=0.25)
savefig("71_04_replay_opportunity_landscape.png", fig)


In [ ]:

# 71.05 — Safety operating slice

safety = df[df.domain == "safety"].copy()
vals = {
    "Replay reward": [
        safety.threshold_reward.mean(),
        safety.learned_reward.mean(),
        safety.always_continue_reward.mean(),
        safety.always_verify_reward.mean()
    ],
    "Safe routing rate": [
        safety.threshold_safe_routing.mean(),
        safety.learned_safe_routing.mean(),
        safety.always_continue_safe_routing.mean(),
        safety.always_verify_safe_routing.mean()
    ],
    "Verifier rate": [
        safety.threshold_verifier.mean(),
        safety.learned_verifier.mean(),
        0.0,
        1.0
    ],
}

fig, ax = plt.subplots(figsize=(14, 7))
ax.set_title("71.05 — Safety operating slice")
x = np.arange(len(vals))
width = 0.18
for i, p in enumerate(policies):
    series = [vals[k][i] for k in vals.keys()]
    ax.bar(x + (i-1.5)*width, series, width, label=labels[i], color=COLORS[p])
    for j, v in enumerate(series):
        ax.text(j + (i-1.5)*width, v + 0.02, f"{v:.2f}", ha="center", fontsize=11)
ax.set_xticks(x)
ax.set_xticklabels(list(vals.keys()))
ax.set_ylim(0, 1.2)
ax.legend(ncol=4, loc="upper center")
ax.grid(True, axis="y", alpha=0.25)
savefig("71_05_safety_operating_slice.png", fig)

# Legacy split-panel version, included so 71.html can use either form.
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
fig.suptitle("71.05 — Safety slice", fontsize=28, y=1.05)
for ax, (metric, series) in zip(axes, vals.items()):
    ax.bar(range(4), series, color=[COLORS[p] for p in policies])
    ax.set_title(metric)
    ax.set_ylim(0, 1.2)
    ax.set_xticks(range(4))
    ax.set_xticklabels(labels, rotation=35, ha="right")
    ax.grid(True, alpha=0.25)
    for i, v in enumerate(series):
        ax.text(i, v+0.02, f"{v:.2f}", ha="center", fontsize=11)
savefig("71_05_safety_slice.png", fig)


In [ ]:

# 71.06 — Budget regimes and replay lift curve

budget_order = ["low_budget", "medium_budget", "high_budget"]
fig, ax = plt.subplots(figsize=(12, 6))
ax.set_title("71.06 — Replay reward by budget regime")
x = np.arange(len(budget_order))
width = 0.18
for i, p in enumerate(policies):
    means = []
    for b in budget_order:
        sub = df[df.budget_regime == b]
        col = {
            "threshold": "threshold_reward",
            "learned": "learned_reward",
            "always_continue": "always_continue_reward",
            "always_verify": "always_verify_reward",
        }[p]
        means.append(sub[col].mean())
    ax.bar(x + (i-1.5)*width, means, width, label=labels[i], color=COLORS[p])
ax.set_xticks(x)
ax.set_xticklabels(["low budget", "medium budget", "high budget"])
ax.set_ylabel("Mean replay reward")
ax.set_ylim(0, 0.95)
ax.legend(ncol=2)
ax.grid(True, axis="y", alpha=0.25)
savefig("71_06_budget_regimes.png", fig)

fractions = np.linspace(0.06, 1.0, 20)
utility_lift = []
regret_reduction = []
risk_change = []
rng = np.random.default_rng(7106)
for f in fractions:
    take = df.sample(frac=f, random_state=int(f*10000))
    utility_lift.append(take.learned_reward.mean() - take.threshold_reward.mean())
    regret_reduction.append(take.threshold_regret.mean() - take.learned_regret.mean())
    risk_change.append(take.threshold_risk_violation.mean() - take.learned_risk_violation.mean())

fig, ax = plt.subplots(figsize=(12, 7))
ax.set_title("71.06 — Replay lift curve")
ax.plot(fractions, utility_lift, marker="o", linewidth=2.5, label="utility lift")
ax.plot(fractions, regret_reduction, marker="o", linewidth=2.5, label="regret reduction")
ax.plot(fractions, risk_change, marker="o", linewidth=2.5, label="risk change")
ax.axhline(0, color="black", linewidth=1)
ax.text(0.36, max(regret_reduction)*0.75, "early replay captures\nmost useful corrections", fontsize=16)
ax.set_xlabel("Replay fraction")
ax.set_ylabel("Change vs threshold policy")
ax.grid(True, alpha=0.25)
ax.legend(loc="lower left")
savefig("71_06_replay_lift_curve.png", fig)


## 71.07–71.10 — Transitions, calibration, failure modes, and deployment gate

In [ ]:

# 71.07 — Policy transition heatmap and map

cm = pd.crosstab(df.threshold_action, df.learned_action).reindex(index=ACTIONS, columns=ACTIONS, fill_value=0)
row_norm = cm.div(cm.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(row_norm.values, vmin=0, vmax=1, cmap="viridis")
ax.set_title("71.07 — Policy transition heatmap")
ax.set_xticks(range(len(ACTIONS)))
ax.set_yticks(range(len(ACTIONS)))
ax.set_xticklabels(ACTIONS, rotation=35, ha="right")
ax.set_yticklabels(ACTIONS)
ax.set_xlabel("Learned policy action")
ax.set_ylabel("Threshold policy action")
for i in range(len(ACTIONS)):
    for j in range(len(ACTIONS)):
        val = cm.iloc[i, j]
        pct = row_norm.iloc[i, j]
        color = "white" if pct > 0.55 else "black"
        ax.text(j, i, f"{val}\n{pct:.0%}", ha="center", va="center", fontsize=11, color=color,
                fontweight="bold" if i == j else "normal")
cb = fig.colorbar(im, ax=ax)
cb.set_label("Row-normalized share")
savefig("71_07_policy_transition_heatmap.png", fig)

# Transition map, less dense than earlier v2: show top flows only.
flows = cm.stack().reset_index()
flows.columns = ["from", "to", "count"]
flows = flows[flows["from"] != flows["to"]].sort_values("count", ascending=False).head(6)

fig, ax = plt.subplots(figsize=(14, 7))
ax.set_axis_off()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title("71.07 — Policy transition map", pad=20)

left_x, right_x = 0.12, 0.78
ys = dict(zip(ACTIONS, np.linspace(0.78, 0.18, len(ACTIONS))))
for a in ACTIONS:
    y = ys[a]
    ax.text(left_x-0.08, y, a, ha="right", va="center", fontsize=17)
    ax.text(right_x+0.20, y, a, ha="left", va="center", fontsize=17)
    ax.add_patch(Rectangle((left_x-0.06, y-0.035), 0.18, 0.07, color=COLORS[a], alpha=0.82))
    ax.add_patch(Rectangle((right_x, y-0.035), 0.18, 0.07, color=COLORS[a], alpha=0.82))

ax.text(left_x, 0.92, "Threshold policy", fontsize=20, fontweight="bold")
ax.text(right_x, 0.92, "Learned policy", fontsize=20, fontweight="bold")

max_count = max(flows["count"].max(), 1)
for _, r in flows.iterrows():
    y1, y2 = ys[r["from"]], ys[r["to"]]
    width = 1 + 8 * r["count"] / max_count
    arrow = FancyArrowPatch((left_x+0.12, y1), (right_x, y2),
                            connectionstyle="arc3,rad=0.18",
                            arrowstyle="-|>", mutation_scale=18,
                            linewidth=width, color=COLORS[r["to"]], alpha=0.45)
    ax.add_patch(arrow)
    ax.text(0.49, (y1+y2)/2, str(int(r["count"])), fontsize=11, ha="center", va="center", alpha=0.8)
ax.text(0.5, 0.06, "width = transition frequency", ha="center", fontsize=14)
savefig("71_07_policy_transition_map.png", fig)

# Domain replay reward heatmap.
domain_order = ["code", "long_context", "math", "qa", "safety"]
mat = []
for d in domain_order:
    sub = df[df.domain == d]
    t = sub.threshold_reward.mean()
    l = sub.learned_reward.mean()
    mat.append([t, l, l-t])
mat = np.array(mat)
fig, ax = plt.subplots(figsize=(10, 7))
ax.set_title("71.07 — Domain replay reward and learned lift")
im = ax.imshow(mat, aspect="auto", cmap="viridis", vmin=min(0, mat.min()), vmax=max(0.8, mat.max()))
ax.set_yticks(range(len(domain_order)))
ax.set_yticklabels(domain_order)
ax.set_xticks(range(3))
ax.set_xticklabels(["threshold", "learned", "lift"])
for i in range(len(domain_order)):
    for j in range(3):
        ax.text(j, i, f"{mat[i,j]:.3f}", ha="center", va="center",
                color="white" if j < 2 else "black", fontsize=13)
cb = fig.colorbar(im, ax=ax)
savefig("71_07_domain_replay_heatmap.png", fig)


In [ ]:

# 71.08 — Policy probability / confidence calibration

learned_prob = df[[f"prob_{a}" for a in ACTIONS]].max(axis=1)
agreement = (df.learned_action == df.best_action).astype(float)

bins = pd.qcut(learned_prob, q=5, duplicates="drop")
cal = pd.DataFrame({"prob": learned_prob, "agree": agreement, "bin": bins}).groupby("bin").agg(
    mean_prob=("prob", "mean"),
    empirical=("agree", "mean"),
    n=("agree", "size")
).reset_index()

fig, ax = plt.subplots(figsize=(9, 8))
ax.set_title("71.08 — Policy confidence calibration")
ax.plot([0, 1], [0, 1], "--", color="0.6", linewidth=2, label="perfect calibration")
ax.plot(cal.mean_prob, cal.empirical, marker="o", linewidth=3, color=COLORS["learned"], label="learned policy")
for _, r in cal.iterrows():
    ax.text(r.mean_prob, min(0.98, r.empirical+0.03), f"{int(r.n)}", ha="center", fontsize=12)
ax.set_xlabel("Mean learned action probability")
ax.set_ylabel("Empirical agreement with replay-best action")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.legend(loc="upper left")
ax.grid(True, alpha=0.25)
savefig("71_08_policy_calibration.png", fig)

# alias name from earlier pass
savefig("71_08_policy_probability_calibration.png", fig)


In [ ]:

# 71.09 — Learned policy failure modes

fail = df[df.learned_action != df.best_action].copy()
fail["mode"] = fail.best_action + " → " + fail.learned_action
mode_counts = fail["mode"].value_counts().head(8)

fig, ax = plt.subplots(figsize=(12, 7))
ax.set_title("71.09 — Learned policy failure modes")
y = np.arange(len(mode_counts))
ax.barh(y, mode_counts.values, color=COLORS["learned"])
ax.set_yticks(y)
ax.set_yticklabels(mode_counts.index)
ax.invert_yaxis()
ax.set_xlabel("Replay states")
ax.set_ylabel("Best action → learned action")
ax.grid(True, axis="x", alpha=0.25)
for i, v in enumerate(mode_counts.values):
    ax.text(v + max(mode_counts.values)*0.015, i, str(v), va="center", fontsize=12)
savefig("71_09_failure_modes.png", fig)

# alias used in some directories
savefig("71_09_learned_policy_failure_modes.png", fig)


In [ ]:

# 71.10 — Deployment readiness gate

gate_items = [
    ("replay utility lifts threshold", df.learned_reward.mean() - df.threshold_reward.mean(), "check"),
    ("verifier rate bounded", df.learned_verifier.mean(), "check" if df.learned_verifier.mean() < 0.25 else "review"),
    ("regret reduced", df.threshold_regret.mean() - df.learned_regret.mean(), "check"),
    ("risk violation bounded", df.learned_risk_violation.mean(), "check" if df.learned_risk_violation.mean() < 0.05 else "review"),
    ("fallback bounded", (df.learned_action == "fallback").mean(), "check" if (df.learned_action == "fallback").mean() < 0.08 else "review"),
    ("online validation", np.nan, "pending"),
]
ready = sum(1 for _, _, s in gate_items if s == "check")
share = ready / len(gate_items)

fig, ax = plt.subplots(figsize=(12, 7))
ax.set_axis_off()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title("71.10 — Deployment readiness gate", pad=20)

for i, (label, val, status) in enumerate(gate_items):
    y = 0.80 - i*0.115
    if status == "check":
        color, mark = COLORS["stop"], "✓"
    elif status == "pending":
        color, mark = COLORS["verify"], "○"
    else:
        color, mark = COLORS["fallback"], "!"
    ax.add_patch(Rectangle((0.08, y-0.035), 0.075, 0.055, color=color, ec="black", lw=1.2))
    ax.text(0.117, y-0.007, mark, ha="center", va="center", color="white", fontsize=20, fontweight="bold")
    ax.text(0.20, y-0.007, label, ha="left", va="center", fontsize=20)
    val_text = "pending" if status == "pending" else f"{val:.3f}"
    ax.text(0.72, y-0.007, val_text, ha="left", va="center", fontsize=18)

ax.text(0.5, 0.06, f"Engineering readiness: REVIEW  ({share:.0%} checks ready)",
        ha="center", va="center", fontsize=24, fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="black", lw=1.5))
savefig("71_10_deployment_readiness_gate.png", fig)

# shorter alias
savefig("71_10_deployment_gate.png", fig)


## Download artifacts

The final cell zips all generated PNGs plus replay CSV/summary JSON. It also includes a Colab `files.download()` helper with the correct zip filename.

In [ ]:

# Export notebook artifacts

csv_path = DATA_DIR / "notebook_71_replay_rows.csv"
summary_path = DATA_DIR / "notebook_71_summary.json"

summary = {
    "rows": int(len(df)),
    "figure_count": len(list(FIG_DIR.glob("*.png"))),
    "mean_threshold_reward": float(df.threshold_reward.mean()),
    "mean_learned_reward": float(df.learned_reward.mean()),
    "mean_threshold_regret": float(df.threshold_regret.mean()),
    "mean_learned_regret": float(df.learned_regret.mean()),
    "learned_verifier_rate": float(df.learned_verifier.mean()),
    "learned_risk_violation_rate": float(df.learned_risk_violation.mean()),
}
summary_path.write_text(json.dumps(summary, indent=2))

zip_name = "Notebook_71_v3_full_figures.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in sorted(os.listdir(FIG_DIR)):
        if fname.endswith(".png"):
            zf.write(FIG_DIR / fname, arcname=f"figures/{fname}")
    zf.write(csv_path, arcname="data/notebook_71_replay_rows.csv")
    zf.write(summary_path, arcname="data/notebook_71_summary.json")

print(f"Created {zip_name}")
display(FileLink(zip_name))

# Colab download helper. Run in Colab only.
try:
    from google.colab import files
    files.download(zip_name)
except Exception:
    print("Colab files.download skipped outside Colab.")
